# Kelvin classic demos: theory and source archive

This notebook promotes the classic Kelvin examples into the docs layer. It gathers the A-formulation, H-formulation, Omega-ReducedOmega, and `radia_iem_vs_fem_sphere.py` scripts that are explanatory demos rather than validation-locked tests.

The notebook intentionally does not rerun the full FEM solves. Those scripts can be expensive and some already carry durable `.png`, `.mat`, `.json`, `.vol`, or `.vtu` artifacts. Instead, the synchronized JSON stores the full source text and SHA-256 hashes so the examples tree does not need to keep every standalone demo script after migration.


## Core convention

For spherical Kelvin inversion with image radius coordinate `rho'`, the repository convention is:

$$\nu_\mathrm{ext}' = (\rho'/R)^2 \nu_0$$

for HCurl A-formulation, and

$$\mu_\mathrm{ext}' = (R/\rho')^2 \mu_0$$

for scalar H/Omega formulations. These factors are reciprocals. The docs source of truth is `docs/kelvin/KELVIN_TRANSFORMATION.md`; the source convention declaration is `docs/kelvin/CONVENTION.md`.

In [1]:
from pathlib import Path
import datetime as dt
import json
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'kelvin_examples_migration.py').exists():
    NOTEBOOK_DIR = Path('docs/kelvin').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from kelvin_examples_migration import (
    build_migration_report,
    build_source_archive,
    markdown_table,
    package_versions,
    write_report_json,
)

archive_path = NOTEBOOK_DIR / 'kelvin_classic_demos_results.json'
migration = build_migration_report()
fresh_archive = build_source_archive(
    migration,
    lanes=['docs_classic_demo_candidate'],
    include_source=True,
)
archive = fresh_archive
if archive_path.exists():
    existing = json.loads(archive_path.read_text(encoding='utf-8'))
    existing_archive = existing.get('source_archive') if isinstance(existing, dict) else None
    existing_count = existing_archive.get('summary', {}).get('archived_files', 0) if isinstance(existing_archive, dict) else 0
    if existing_count > fresh_archive['summary']['archived_files']:
        archive = existing_archive
        print('loaded existing classic archive to preserve deleted source snapshots')

classic = archive['files']
classic_by_group = {}
for row in classic:
    classic_by_group.setdefault(row['top_group'], []).append(row)

selected_paths = [
    'examples/kelvin_transformation/A-formulation/A_formulation_sphere_with_Kelvin.py',
    'examples/kelvin_transformation/H-formulation/3D_dipole_with_Kelvin.py',
    'examples/kelvin_transformation/Omega_ReducedOmega/Sphere/3D_sphere_with_Kelvin.py',
    'examples/kelvin_transformation/radia_iem_vs_fem_sphere.py',
]
archive_by_path = {row['path']: row for row in classic}
excerpts = []
for rel_path in selected_paths:
    row = archive_by_path.get(rel_path)
    if row is None:
        continue
    text = row['source'].get('source_text', '')
    lines = text.splitlines()
    excerpts.append({
        'path': rel_path,
        'sha256': row['source']['sha256'],
        'line_count': row['source']['line_count'],
        'excerpt_line_count': min(120, len(lines)),
        'excerpt': '\n'.join(lines[:120]),
    })

payload = {
    'schema': 'radia.docs.kelvin_classic_demos.v2',
    'generated_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'versions': package_versions(),
    'source_migration_json': 'docs/kelvin/kelvin_examples_migration_results.json',
    'summary': {
        'classic_demo_python_files': archive['summary']['archived_files'],
        'archived_lines': archive['summary']['archived_lines'],
        'archived_bytes': archive['summary']['archived_bytes'],
        'groups': archive['summary']['by_group'],
        'selected_source_excerpts': len(excerpts),
        'full_source_archive': True,
    },
    'classic_demo_files': [
        {
            'path': row['path'],
            'group': row['top_group'],
            'bytes': row['source']['bytes'],
            'line_count': row['source']['line_count'],
            'sha256': row['source']['sha256'],
            'sibling_artifacts': row['sibling_artifacts'],
        }
        for row in classic
    ],
    'source_excerpts': excerpts,
    'source_archive': archive,
}
write_report_json(payload, archive_path)
print(json.dumps(payload['summary'], ensure_ascii=False, indent=2))
print(f'\nwrote {archive_path.relative_to(NOTEBOOK_DIR)}')


loaded existing classic archive to preserve deleted source snapshots
{
  "classic_demo_python_files": 37,
  "archived_lines": 15663,
  "archived_bytes": 628228,
  "groups": {
    "H-formulation": 15,
    "A-formulation": 13,
    "Omega_ReducedOmega": 8,
    "radia_iem_vs_fem_sphere.py": 1
  },
  "selected_source_excerpts": 4,
  "full_source_archive": true
}

wrote kelvin_classic_demos_results.json


In [2]:
from IPython.display import Markdown, display

display(Markdown('## Classic demo files by group'))
group_rows = [{'group': group, 'python_files': len(rows)} for group, rows in classic_by_group.items()]
display(Markdown(markdown_table(group_rows, ['group', 'python_files'], max_rows=20)))

display(Markdown('## Classic demo file list'))
file_rows = [
    {
        'path': row['path'],
        'group': row['top_group'],
        'bytes': row['source']['bytes'],
        'lines': row['source']['line_count'],
        'sha256': row['source']['sha256'][:12],
        'artifacts': ', '.join(row['sibling_artifacts'][:4]),
    }
    for row in classic
]
display(Markdown(markdown_table(file_rows, ['path', 'group', 'bytes', 'lines', 'sha256', 'artifacts'], max_rows=45)))


## Classic demo files by group

| group | python_files |
| --- | --- |
| A-formulation | 13 |
| H-formulation | 15 |
| Omega_ReducedOmega | 8 |
| radia_iem_vs_fem_sphere.py | 1 |

## Classic demo file list

| path | group | bytes | lines | sha256 | artifacts |
| --- | --- | --- | --- | --- | --- |
| examples/kelvin_transformation/A-formulation/A_formulation_sphere_simple.py | A-formulation | 16006 | 450 | a335d4045a43 |  |
| examples/kelvin_transformation/A-formulation/A_formulation_sphere_with_Kelvin.py | A-formulation | 30388 | 758 | be85b850314c | A_formulation_sphere_with_Kelvin.png |
| examples/kelvin_transformation/A-formulation/Coil_3D_A_HCurl_PEEC_source.py | A-formulation | 18159 | 461 | a0d5ab3f4359 |  |
| examples/kelvin_transformation/A-formulation/Coil_3D_A_HCurl_with_Kelvin.py | A-formulation | 3669 | 107 | 600f33f9d0dc |  |
| examples/kelvin_transformation/A-formulation/Coil_3D_A_PEEC_step0.py | A-formulation | 6361 | 170 | 846b67b5d729 |  |
| examples/kelvin_transformation/A-formulation/Coil_3D_baseline_via_helpers.py | A-formulation | 5482 | 149 | e470f0eb95e3 |  |
| examples/kelvin_transformation/A-formulation/Coil_A_formulation_simple.py | A-formulation | 19469 | 506 | 3dd0479b9406 |  |
| examples/kelvin_transformation/A-formulation/Coil_A_formulation_with_Kelvin.py | A-formulation | 21066 | 576 | c47290ac445c | Coil_A_formulation_with_Kelvin.png, Coil_A_formulation_with_Kelvin.mat |
| examples/kelvin_transformation/A-formulation/compare_mesh_vs_peec_same_mesh.py | A-formulation | 11178 | 275 | 242075380d73 |  |
| examples/kelvin_transformation/A-formulation/M3_validate_kelvin_driver.py | A-formulation | 3926 | 117 | b052a02f8879 |  |
| examples/kelvin_transformation/A-formulation/ParallelWires_2D_A_formulation_with_Kelvin.py | A-formulation | 21919 | 613 | 622dbf7acf00 |  |
| examples/kelvin_transformation/A-formulation/sphere_in_uniform_field.py | A-formulation | 9581 | 262 | 04043ab5d3c3 |  |
| examples/kelvin_transformation/A-formulation/test_nu_convention.py | A-formulation | 6945 | 189 | 36c327f10c35 |  |
| examples/kelvin_transformation/H-formulation/2D_dipole.py | H-formulation | 17963 | 410 | 90cbe2a70222 |  |
| examples/kelvin_transformation/H-formulation/2D_dipole_half_with_Kelvin.py | H-formulation | 10901 | 295 | b237c0a7110b |  |
| examples/kelvin_transformation/H-formulation/2D_dipole_with_Kelvin.py | H-formulation | 26725 | 629 | 357e9d5b740b | 2D_dipole_with_Kelvin.png, 2D_dipole_with_Kelvin.mat |
| examples/kelvin_transformation/H-formulation/2D_quadrupole.py | H-formulation | 23635 | 531 | 9265e6ec6fec |  |
| examples/kelvin_transformation/H-formulation/2D_quadrupole_with_Kelvin.py | H-formulation | 27696 | 642 | ff426cbbc659 | 2D_quadrupole_with_Kelvin.png, 2D_quadrupole_with_Kelvin.mat |
| examples/kelvin_transformation/H-formulation/3D_dipole.py | H-formulation | 18242 | 433 | d7a4e4d60d85 |  |
| examples/kelvin_transformation/H-formulation/3D_dipole_with_Kelvin.py | H-formulation | 37750 | 884 | 8853acc49a5c | 3D_dipole_with_Kelvin.png, 3D_dipole_with_Kelvin.mat |
| examples/kelvin_transformation/H-formulation/3D_quadrupole.py | H-formulation | 20150 | 475 | 1ef0930484d3 |  |
| examples/kelvin_transformation/H-formulation/3D_quadrupole_with_Kelvin.py | H-formulation | 42080 | 944 | c596aea0ed81 | 3D_quadrupole_with_Kelvin.png, 3D_quadrupole_with_Kelvin.mat |
| examples/kelvin_transformation/H-formulation/Axisymmetric_dipole.py | H-formulation | 22509 | 533 | 4def2fb05830 |  |
| examples/kelvin_transformation/H-formulation/Axisymmetric_dipole_with_Kelvin.py | H-formulation | 30454 | 692 | d7084350ffb2 | Axisymmetric_dipole_with_Kelvin.png, Axisymmetric_dipole_with_Kelvin.mat |
| examples/kelvin_transformation/H-formulation/Fig_1.py | H-formulation | 2195 | 74 | a6f24bb1be18 |  |
| examples/kelvin_transformation/H-formulation/Fig_2.py | H-formulation | 2223 | 74 | f6310e6ce188 |  |
| examples/kelvin_transformation/H-formulation/Fig_3.py | H-formulation | 2318 | 75 | 871afef50b8b |  |
| examples/kelvin_transformation/H-formulation/Laplace3D_dipole_with_Kelvin.py | H-formulation | 21876 | 517 | b890ad079677 | Laplace3D_dipole_with_Kelvin.png |
| examples/kelvin_transformation/Omega_ReducedOmega/convergence_study.py | Omega_ReducedOmega | 7009 | 193 | bec77eaf93cb |  |
| examples/kelvin_transformation/Omega_ReducedOmega/Cylinder/3D_cylinder_with_Kelvin.py | Omega_ReducedOmega | 19231 | 529 | 047980e462a1 | 3D_cylinder_with_Kelvin.png |
| examples/kelvin_transformation/Omega_ReducedOmega/Cylinder/3D_cylinder_with_Kelvin_1_8.py | Omega_ReducedOmega | 20027 | 552 | 01c295cb299a | 3D_cylinder_with_Kelvin_1_8.png |
| examples/kelvin_transformation/Omega_ReducedOmega/Cylinder/Axisymmetric_cylinder_with_Kelvin.py | Omega_ReducedOmega | 22809 | 611 | 3423329caacb | Axisymmetric_cylinder_with_Kelvin.png |
| examples/kelvin_transformation/Omega_ReducedOmega/Omega_ReducedOmega.py | Omega_ReducedOmega | 7540 | 217 | 012f68e3a988 |  |
| examples/kelvin_transformation/Omega_ReducedOmega/reference_2d_axisym.py | Omega_ReducedOmega | 7064 | 216 | 0bc73a50254d |  |
| examples/kelvin_transformation/Omega_ReducedOmega/Sphere/3D_sphere_with_Kelvin.py | Omega_ReducedOmega | 30826 | 739 | 32b3e400d403 | 3D_sphere_with_Kelvin.png |
| examples/kelvin_transformation/Omega_ReducedOmega/Sphere/Axisymmetric_sphere_with_Kelvin.py | Omega_ReducedOmega | 24843 | 617 | 88c334e4f1f4 | Axisymmetric_sphere_with_Kelvin.png |
| examples/kelvin_transformation/radia_iem_vs_fem_sphere.py | radia_iem_vs_fem_sphere.py | 8013 | 148 | eb47838ce8f5 | radia_iem_vs_fem_sphere.json |

In [3]:
for item in excerpts:
    display(Markdown(f"## Source excerpt: `{item['path']}`"))
    display(Markdown('```python\n' + item['excerpt'] + '\n```'))

## Source excerpt: `examples/kelvin_transformation/A-formulation/A_formulation_sphere_with_Kelvin.py`

```python
"""
Axisymmetric A-formulation for magnetic sphere with Z-OFFSET Kelvin transformation

A-formulation for axisymmetric problems:
  Variable: u = r * A_theta (modified vector potential)
  Equation: -div(nu/r * grad(u)) = 0 (no current source)

Relation to B field:
  Br = -(1/r) * du/dz
  Bz = (1/r) * du/dr

Problem: Magnetic sphere (mu_r) in uniform external field H0 (z-direction)

Z-offset Kelvin transformation:
  - Interior domain: half-circle centered at (0, 0), radius a
  - Exterior domain: half-circle centered at (0, z_offset), radius a
  - r = x is the SAME in both domains (key simplification!)
  - Kelvin factor for nu: (rho'/a)^2

Analytical solution for sphere:
  Interior: Hz = 3/(mu_r + 2) * H0  (uniform)
  Exterior: perturbation field decays as 1/r^3
"""
import os
from numpy import *
from ngsolve import *
from netgen.occ import *

print("="*60)
print("Axisymmetric A-formulation with Z-OFFSET Kelvin")
print("Magnetic Sphere in Uniform Field")
print("="*60)

# ============================================================
# Parameters
# ============================================================
R_sphere = 0.5          # Sphere radius [m]
a = 1.0                 # Kelvin boundary radius [m]
z_offset = 2.5          # Z-offset for exterior domain [m]
maxh = 0.03             # Mesh size [m]

mu0 = 4*pi*1e-7         # Vacuum permeability [H/m]
nu0 = 1/mu0             # Vacuum reluctivity [m/H]
mu_r = 100              # Relative permeability
H0 = 1.0                # Applied field [A/m]
B0 = mu0 * H0           # Applied flux density [T]

print(f"\nParameters:")
print(f"  Sphere radius: {R_sphere} m")
print(f"  Kelvin boundary radius: a = {a} m")
print(f"  Z-offset: {z_offset} m")
print(f"  Relative permeability: {mu_r}")
print(f"  Applied field: H0 = {H0} A/m")
print(f"  Applied flux density: B0 = {B0:.6e} T")

# ============================================================
# Analytical Solution
# ============================================================
Hz_int_analytical = 3.0 / (mu_r + 2) * H0
Hz_pert_int = Hz_int_analytical - H0  # Perturbation field inside

print(f"\nAnalytical solution (3D sphere):")
print(f"  Interior Hz = {Hz_int_analytical:.6f} A/m")
print(f"  Interior Hz_pert = {Hz_pert_int:.6f} A/m")

# ============================================================
# Geometry with Z-offset Kelvin (using full circle -> cutter pattern)
# ============================================================
print("\nCreating geometry with Z-offset Kelvin...")

# Interior domain: FULL circle first, then cut to half
wp1 = WorkPlane()
inner_full = wp1.Circle(a).Face()
inner_full.name = "air_inner"

# Magnetic sphere: FULL circle, then cut
wp2 = WorkPlane()
sphere_full = wp2.Circle(R_sphere).Face()
sphere_full.name = "magnetic"
sphere_full.maxh = maxh/2

# Exterior domain: FULL circle at (0, z_offset), then cut
wp3 = WorkPlane(Axes((0, z_offset, 0), n=Z, h=X))
outer_full = wp3.Circle(a).Face()
outer_full.name = "air_outer"

# GND point at center of exterior domain (represents infinity)
gnd_point = Vertex(Pnt(0, z_offset, 0))
gnd_point.name = "GND"

# Cut to get half-circles (keep r >= 0, i.e., x >= 0)
cutter_left = MoveTo(-a-0.1, -a-0.1).Rectangle(a+0.1, 2*a+0.2).Face()
cutter_left_ext = MoveTo(-a-0.1, z_offset - a - 0.1).Rectangle(a+0.1, 2*a+0.2).Face()

inner_half = inner_full - cutter_left
sphere_half = sphere_full - cutter_left
outer_half = outer_full - cutter_left_ext

# Inner air = inner half - sphere half
inner_air = inner_half - sphere_half
inner_air.name = "air_inner"

# ============================================================
# Name boundaries and identify periodic edges
# ============================================================
print("  Naming boundaries...")

# Inner air edges - find Kelvin arcs using vertex distance
kelvin_inner_edges = []
for edge in inner_air.edges:
    cx = edge.center.x
    cy = edge.center.y
    # Check if both vertices are at distance a from origin
    try:
        v0, v1 = edge.vertices
        d0 = sqrt(v0.p.x**2 + v0.p.y**2)
        d1 = sqrt(v1.p.x**2 + v1.p.y**2)
        is_kelvin_arc = abs(d0 - a) < 0.01 and abs(d1 - a) < 0.01 and cx > 0.01
    except:
        is_kelvin_arc = False
```

## Source excerpt: `examples/kelvin_transformation/H-formulation/3D_dipole_with_Kelvin.py`

```python
"""
H-formulation for magnetostatics with perturbation potential
Geometry created internally using OCC
Updated: 2026-02-19

TEST RESULTS SUMMARY (mu_r = 100):
================================

Configuration: Kelvin transformation with periodic BC
  - Interior domain: magnetic sphere (r < 0.5m) + air_inner (0.5m < r < 1.0m)
  - Exterior domain: air_outer (r' < 1.0m), Kelvin-mapped, centered at offset_x = 3.0m
  - Periodic BC between interior (r=R) and exterior (r'=R) at R = 1.0m
  - GND (Dirichlet) at exterior center (r'=0, maps to r=infinity)
  - Modulated permeability in exterior: mu'(r') = (R/r')^2 · mu_0

Results:
  - Origin (0,0,0):   Hz = -0.970588, analytical = -0.970588, error = 0.000%
  - (0.7,0,0):        Hz = -0.353193, analytical = -0.353713, error = 0.147%
  - Interior RMS error (|x| < 0.5m): 9.13e-6 (0.001%)
  - Exterior RMS error (|x| >= 0.5m): 6.43e-4

Periodic BC verification:
  - FreeDofs reduced from 6,309,920 to 6,167,961 (141,959 DOFs coupled)
  - CG solver converged in 355 iterations
"""
import os, sys
from numpy import *
from ngsolve import *
import ngsolve
from ngsolve import TaskManager

# Import OCC geometry
from netgen.occ import *

print("="*60)
print("H-formulation 3D - OCC Geometry")
print("="*60)

# ============================================================
# Geometry Definition (OCC)
# ============================================================
print("\nCreating geometry...")

# Parameters
sphere_radius = 0.5  # Magnetic sphere radius [m]
kelvin_radius = 1.0  # Kelvin transformation radius [m]
maxh_fine = 0.03     # Fine mesh size [m] (for magnetic sphere and inner air)
plot_range = 1.1    # Plot range [m]

# ===== INTERIOR DOMAIN (center at origin) =====
# Magnetic circle
mag_sphere = Sphere(Pnt(0, 0, 0), sphere_radius)
mag_sphere.mat("magnetic")
mag_sphere.maxh = maxh_fine

# Offset for exterior domain (placed separately)
offset_x = 3.0  # Offset to place exterior domain away from interior

# Inner air domain (circle_radius < r < kelvin_radius)
inner_sphere = Sphere(Pnt(0, 0, 0), kelvin_radius)
inner_sphere.maxh = maxh_fine
# Name Kelvin boundary face before subtraction (survives Boolean ops)
for face in inner_sphere.faces:
    face.name = "kelvin_int"
inner_air = inner_sphere - mag_sphere
inner_air.mat("air_inner")

# ===== EXTERIOR DOMAIN (center at offset position) =====
# Outer boundary sphere (no cutoff sphere - solid domain)
outer_sphere = Sphere(Pnt(offset_x, 0, 0), kelvin_radius)
outer_sphere.maxh = maxh_fine
# Name Kelvin boundary face before Glue (survives Boolean ops)
for face in outer_sphere.faces:
    face.name = "kelvin_ext"
outer_sphere.mat("air_outer")

# GND vertex at center (represents r'=0, which maps to r=infinity)
vertex = Vertex(Pnt(offset_x, 0, 0))
vertex.name = "GND"

# Glue all domains
geo = Glue([inner_air, mag_sphere, outer_sphere, vertex])

# ===== NAME THE FACES AND EDGES =====
print("\nNaming faces and edges...")
print(f"  Number of faces: {len(geo.faces)}")

# Name the solids
geo.solids[0].name = "air_inner"
geo.solids[1].name = "magnetic"
geo.solids[2].name = "air_outer"

print("\nIdentifying periodic boundaries...")
# Find Kelvin boundary faces by name (named during geometry construction)
print(f"  Number of solids: {len(geo.solids)}")
for i, solid in enumerate(geo.solids):
    print(f"  Solid[{i}] ({solid.name}): {len(solid.faces)} faces")

kelvin_int_face = None
kelvin_ext_face = None
for solid in geo.solids:
    for face in solid.faces:
        if face.name == "kelvin_int":
            kelvin_int_face = face
            print(f"  Found kelvin_int face in solid '{solid.name}'")
        elif face.name == "kelvin_ext":
            kelvin_ext_face = face
            print(f"  Found kelvin_ext face in solid '{solid.name}'")

if kelvin_int_face is not None and kelvin_ext_face is not None:
    kelvin_int_face.Identify(kelvin_ext_face, "periodic", IdentificationType.PERIODIC)
    print("  Periodic identification applied between kelvin_int and kelvin_ext")
else:
    raise RuntimeError(f"Could not find Kelvin boundary faces! "
                       f"(kelvin_int: {kelvin_int_face is not None}, kelvin_ext: {kelvin_ext_face is not None})")

# ============================================================
# Mesh Generation
# ============================================================
print("\nGenerating mesh...")
```

## Source excerpt: `examples/kelvin_transformation/Omega_ReducedOmega/Sphere/3D_sphere_with_Kelvin.py`

```python
"""
Omega-Reduced Omega Method for 3D Magnetostatics with Kelvin Transformation
Problem: Magnetic sphere (mu_r=100) in uniform z-directed background field

Based on Omega_ReducedOmega.py implementation:
- Sign convention: B = mu * grad(Omega), H = grad(Omega)
- Source potential: Omega_s = H0 * z (so grad(Omega_s) = H_s)
- Total region (magnetic sphere): No source term in weak form
- Reduced region (air): Source term from Omega_s

Kelvin transformation:
- Maps infinite exterior domain to finite sphere
- Permeability transformation: mu'(r') = (R/r')^2 * mu0
- Periodic BC couples interior (r=R) with exterior (r'=R)
- Dirichlet BC at exterior center (r'=0 -> r=infinity): Omega = 0

IMPORTANT: 3D Normal Direction
- In NGSolve 3D, specialcf.normal(mesh.dim) on the sphere boundary points INWARD
  (from air into magnetic sphere), but Omega-Reduced Omega requires OUTWARD normal.
- Therefore, we NEGATE the normal: normal = -specialcf.normal(mesh.dim)
- This is in contrast to 2D axisymmetric where the normal already points outward.

Analytical solution:
- Inside sphere: Hz = 3/(mu_r + 2) * H0
- Outside sphere: dipole + uniform field

Perturbation field energy:
- Interior: W_in = (1/2) * mu_r * mu0 * [(mu_r-1)/(mu_r+2)]^2 * H0^2 * V_sphere
- Exterior: W_out = mu0 * m^2 / (12*pi*a^3), where m = 4*pi*a^3*(mu_r-1)/(mu_r+2)*H0

Author: Claude Code
Date: 2025-12-23
"""
import os
from numpy import *
from ngsolve import *
from ngsolve import TaskManager
from netgen.occ import *

print("=" * 60)
print("Omega-Reduced Omega Method - 3D Magnetic Sphere with Kelvin Transform")
print("=" * 60)

# ============================================================
# Parameters
# ============================================================
sphere_radius = 0.5      # Magnetic sphere radius [m]
kelvin_radius = 1.0      # Kelvin transformation radius [m]
mu_r = 100               # Relative permeability
mu0 = 4 * pi * 1e-7      # Vacuum permeability [H/m]

# Source field: H_s = (0, 0, H0) uniform in z-direction
H0 = 1.0  # [A/m]

# Mesh parameters
maxh_sphere = 0.05       # Finer mesh for magnetic sphere
maxh_air = 0.08          # Moderate mesh for air region
fe_order = 2             # Finite element order

# Offset for exterior domain (z-direction to match axisymmetric)
offset_z = 3.0

print(f"\nProblem parameters:")
print(f"  Sphere radius: {sphere_radius} m")
print(f"  Kelvin radius: {kelvin_radius} m")
print(f"  Relative permeability: mu_r = {mu_r}")
print(f"  Source field: H_s = (0, 0, {H0}) A/m")

# ============================================================
# Geometry Definition (OCC)
# ============================================================
print("\nCreating geometry using OCC...")

# Interior domain: magnetic sphere at origin
mag_sphere = Sphere(Pnt(0, 0, 0), sphere_radius)
mag_sphere.mat("magnetic")
mag_sphere.maxh = maxh_sphere
# Name the magnetic sphere surface
for face in mag_sphere.faces:
    face.name = "sphere"

# Inner air domain (annulus between magnetic sphere and Kelvin boundary)
inner_sphere = Sphere(Pnt(0, 0, 0), kelvin_radius)
inner_sphere.maxh = maxh_air
# Name the Kelvin boundary (interior side)
for face in inner_sphere.faces:
    face.name = "kelvin_int"

inner_air = inner_sphere - mag_sphere
inner_air.mat("air_inner")

# Exterior domain (Kelvin-transformed, centered at z-offset position)
outer_sphere = Sphere(Pnt(0, 0, offset_z), kelvin_radius)
outer_sphere.maxh = maxh_air
outer_sphere.mat("air_outer")
# Name the Kelvin boundary (exterior side)
for face in outer_sphere.faces:
    face.name = "kelvin_ext"

# GND vertex at center of exterior domain (represents r'=0 -> r=infinity)
vertex = Vertex(Pnt(0, 0, offset_z))
vertex.name = "GND"

# Glue all domains
geo = Glue([inner_air, mag_sphere, outer_sphere, vertex])

# Name the solids
geo.solids[0].name = "air_inner"
geo.solids[1].name = "magnetic"
geo.solids[2].name = "air_outer"

# ===== IDENTIFY PERIODIC FACES =====
print("\nIdentifying periodic boundaries...")

# Print solid and face information
print(f"  Number of solids: {len(geo.solids)}")
for i, solid in enumerate(geo.solids):
    print(f"  Solid[{i}] ({solid.name}): {len(solid.faces)} faces")
    for j, face in enumerate(solid.faces):
        print(f"    Face[{j}]: name='{face.name}'")
```

## Source excerpt: `examples/kelvin_transformation/radia_iem_vs_fem_sphere.py`

```python
"""3-way validation: Radia open-boundary IEM vs reduced-Omega + Kelvin FEM vs the analytic sphere.

A linear magnetic sphere (relative permeability mu_r) in a uniform applied field H0 has the EXACT
uniform interior field

    H_in = 3 / (mu_r + 2) * H0 ,   M = (mu_r - 1) * H_in   (magnetic-sphere-in-uniform-field)

Both independent solvers are checked against that analytic answer on the SAME quarter-sphere geometry
(`Cubit_1_4_p_convergence/sphere_1_4_p{1,2,3}.vol`, x>=0 & y>=0, radius 50 mm), order-matched:

  * analytic   : 3 / (mu_r + 2)
  * Radia IEM  : MMM -- the tetrahedron surface-charge / dipole OPEN-BOUNDARY integral method (no air
                 mesh, exact analytic open boundary).  The sphere meshes as tets, so MMM is the Radia IEM
                 member here; six-face surface-charge MSC is the HEXAHEDRON variant of the same surface-charge IEM
                 (parity yano == MMM == HDiv-VIM locked by tests/feec/parity_vs_msc, cube 0.76%), and the
                 loop removal (rad.GetLoopBasis, Stage 1) is a field-PRESERVING yano-hex conditioning fix
                 that does not change this accuracy.  Quarter model solved with image='+x+y' (the z-field
                 is parallel to both the x=0 and y=0 planes -> symmetric; MMM has NO IMA boundary
                 limitation, CLAUDE.md).  Observable = volume-averaged magnetization <M_z> ->
                 H_in = <M_z>/(mu_r-1); NOT the inside-iron field eval (an unreliable dipole approximation
                 for MMM, per CLAUDE.md "rad.Fld inside materials").
  * FEM        : NGSolve reduced-Omega (2-scalar) + Kelvin transformation (open boundary, no PML), via
                 solve_kelvin_benchmark on the full magnetic+air+kelvin mesh.

TWO lessons the p-sweep makes explicit (this is what to get right with reduced-Omega):
  1. reduced-Omega MUST use p>=2.  p=1 here is +7.8% (unusable).
  2. The FE order MUST be MATCHED to the mesh geometry-curving order: solve order p on the order-p
     curved mesh.  Running p=3 on an order-2-curved mesh is a geometry mismatch and gives a spurious
     result that does not reflect true p=3.  Order-matched, the residual at p=2/3 (~0.5-0.7%, == the
     locked golden bands) is the coarse academic mesh's h + Kelvin-truncation floor -- NOT FE-order and
     NOT a code bug.  The no-air-mesh IEM lands closest.
"""
import json
import math
import os
import sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
REPO = os.path.abspath(os.path.join(HERE, "..", ".."))
DMESH = os.path.join(HERE, "Cubit_1_4_p_convergence")
sys.path.insert(0, os.path.join(REPO, "src", "radia"))
sys.path.insert(0, os.path.join(REPO, "src", "radia", "panels"))

import ngsolve as ng  # noqa: E402
import radia as rad  # noqa: E402
from netgen_mesh_import import extract_elements  # noqa: E402
from calc_kelvin_benchmark import solve_kelvin_benchmark  # noqa: E402

MU_R = 100.0
MU0 = 4.0e-7 * math.pi


def mesh_for(p):
    return os.path.join(DMESH, f"sphere_1_4_p{p}.vol")


def analytic_ratio(mu_r):
    return 3.0 / (mu_r + 2.0)


def radia_iem_mmm(mu_r, H0=1000.0):
    """Radia MMM (tet surface-charge IEM, analytic open boundary) on the x>=0,y>=0 quarter sphere +
    image='+x+y'.  Returns H_in/H0 from the volume-averaged magnetization (the reliable MMM observable).
    MMM uses straight tets (no curving), so the order-1 mesh vertices are used."""
    mesh = ng.Mesh(mesh_for(1))
    els, _ = extract_elements(mesh, material_filter="magnetic", allow_hex=False)
    V = [np.array(e["vertices"], float) for e in els]
    vol = np.array([abs(np.dot(p[1] - p[0], np.cross(p[2] - p[0], p[3] - p[0]))) / 6.0 for p in V])
    cen = np.array([p.mean(0) for p in V])

    rad.UtiDelAll()
    rad.set_demag_backend("auto")        # tet -> MMM (C++)
    objs = []
    for p in V:
        t = rad.ObjTetrahedron([list(v) for v in p], [0, 0, 0])
        rad.MatApl(t, rad.MatLin(mu_r))
        objs.append(t)
    cont = rad.ObjCnt(objs + [rad.ObjBckg(lambda q: [0, 0, MU0 * H0])])
    rad.Solve(cont, 1e-6, 3000, 0, image="+x+y")    # z-field // x=0 and y=0 planes -> symmetric
    Mz = np.array([rad.Fld(cont, "m", list(c))[2] for c in cen])
    rad.UtiDelAll()
    Mz_avg = float((Mz * vol).sum() / vol.sum())
    return Mz_avg / (mu_r - 1.0) / H0, Mz_avg, len(els)


def reduced_omega_kelvin_fem(mu_r, H0=1.0, orders=(1, 2, 3)):
    """ORDER-MATCHED reduced-Omega + Kelvin p-sweep: solve order p on the order-p curved mesh."""
    ng.SetNumThreads(4)
    out = []
    for p in orders:
        with ng.TaskManager():
            r = solve_kelvin_benchmark(mesh_for(p), mu_r=mu_r, H0=H0, field_axis="z",
                                       fes_order=p, R_kelvin=0.20)
        if "error" in r:
            raise RuntimeError(f"p={p}: {r['error']}")
        out.append({"fes_order": p, "ratio_Hin_over_H0": r["Hi_origin"] / H0, "ndof": r.get("ndof")})
    return out


def main():
    ratio_ana = analytic_ratio(MU_R)
    ratio_mmm, Mz_avg, n_tet = radia_iem_mmm(MU_R)
    fem = reduced_omega_kelvin_fem(MU_R, orders=(1, 2, 3))

    print(f"\nMagnetic sphere in uniform field, mu_r={MU_R:.0f}  (analytic H_in/H0 = {ratio_ana:.6f})")
    print("quarter sphere, ORDER-MATCHED (geometry curving order == FE order)\n")
    print(f"  {'method':<40} {'H_in/H0':>10} {'err vs analytic':>16}   note")
    print(f"  {'analytic 3/(mu_r+2)':<40} {ratio_ana:>10.6f} {0.0:>+15.2%}")
    print(f"  {'Radia IEM (MMM, tet, no air mesh)':<40} {ratio_mmm:>10.6f} "
          f"{ratio_mmm/ratio_ana-1.0:>+15.2%}   {n_tet} tets, image=+x+y")
    for f in fem:
        note = "(p=1 UNRELIABLE -- use p>=2)" if f["fes_order"] == 1 else f"ndof={f['ndof']}, order-matched"
        print(f"  {'reduced-Omega + Kelvin FEM p=' + str(f['fes_order']):<40} "
              f"{f['ratio_Hin_over_H0']:>10.6f} {f['ratio_Hin_over_H0']/ratio_ana-1.0:>+15.2%}   {note}")

    out = {
        "geometry": "magnetic sphere (R=50mm) in uniform field; quarter model, order-matched p-meshes",
        "meshes": [os.path.relpath(mesh_for(p), REPO).replace("\\", "/") for p in (1, 2, 3)],
```

## Promotion decision

The classic scripts are now represented by this result-bearing notebook plus `kelvin_classic_demos_results.json`. The JSON carries full source text and SHA-256 hashes for every archived script, while the notebook keeps the theory route and representative excerpts readable.

Validation-named scripts, Cubit p-convergence samples, DtN/API candidates, and higher-level adaptive drivers remain outside this prune lane until they are separately lifted into `validation_test`, `src`, or a focused docs notebook.
